# XAI Temporal Stability - Simplified Notebook

This version keeps only the core workflow:
1. Load and clean LendingClub data
2. Train one frozen XGBoost model on base years
3. Compute SHAP attributions
4. Compute TESI across future windows
5. Compare TESI vs AUC over time

Removed for clarity and speed: LightGBM, LIME, KernelSHAP, Optuna retraining, interaction drift, EWMA, and advanced hypothesis test blocks.

In [2]:
# Core imports and reproducibility
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import shap

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Environment ready")
print("xgboost:", xgb.__version__)
print("shap:", shap.__version__)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Locate LendingClub file (Kaggle first, then local fallbacks)
CANDIDATE_PATHS = [
    "/kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz",
    "/kaggle/input/lending-club/accepted_2007_to_2018q4.csv.gz",
    "accepted_2007_to_2018Q4.csv.gz",
    "accepted_2007_to_2018Q4.csv",
]

data_file = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if data_file is None and os.path.exists("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            lower = f.lower()
            if "accepted" in lower and (lower.endswith(".csv") or lower.endswith(".csv.gz")):
                data_file = os.path.join(root, f)
                break
        if data_file:
            break

if data_file is None:
    raise FileNotFoundError(
        "LendingClub dataset not found. Add dataset in Kaggle or place CSV locally."
    )

print("Loading:", data_file)
raw_df = pd.read_csv(data_file, low_memory=False)
print("Raw shape:", raw_df.shape)

In [ ]:
# Minimal preprocessing
FEATURE_NAMES = [
    "loan_amnt", "int_rate", "annual_inc", "dti",
    "revol_util", "open_acc", "total_acc", "pub_rec",
]

STATUS_MAP = {"Fully Paid": 0, "Charged Off": 1}
df = raw_df[raw_df["loan_status"].isin(STATUS_MAP)].copy()
df["target"] = df["loan_status"].map(STATUS_MAP)

issue_col = "issue_d" if "issue_d" in df.columns else "issue_date"
df[issue_col] = pd.to_datetime(df[issue_col], format="mixed", errors="coerce")
df["issue_year"] = df[issue_col].dt.year

for col in FEATURE_NAMES:
    if df[col].dtype == "object":
        cleaned = df[col].astype(str).str.replace(r"[%$,\s]", "", regex=True)
        df[col] = pd.to_numeric(cleaned, errors="coerce")

df = df[FEATURE_NAMES + ["target", "issue_year"]]
df = df[df["issue_year"].between(2013, 2017)]
df = df.dropna().reset_index(drop=True)

income_cap = df["annual_inc"].quantile(0.999)
df = df[df["annual_inc"] <= income_cap].reset_index(drop=True)

print("Prepared shape:", df.shape)
print(df["issue_year"].value_counts().sort_index())

In [ ]:
# Chronological windows
WINDOWS = {
    "T_train": [2013, 2014],
    "T1": [2015],
    "T2": [2016],
    "T3": [2017],
}
WINDOW_ORDER = list(WINDOWS.keys())
BASE_WINDOW = WINDOW_ORDER[0]

X_splits = {}
y_splits = {}
for window_name, years in WINDOWS.items():
    split_df = df[df["issue_year"].isin(years)]
    X_splits[window_name] = split_df[FEATURE_NAMES].to_numpy()
    y_splits[window_name] = split_df["target"].to_numpy()

scaler = StandardScaler()
X_splits[BASE_WINDOW] = scaler.fit_transform(X_splits[BASE_WINDOW])
for window_name in WINDOW_ORDER[1:]:
    X_splits[window_name] = scaler.transform(X_splits[window_name])

for window_name in WINDOW_ORDER:
    y = y_splits[window_name]
    print(f"{window_name}: n={len(y):,}, default_rate={y.mean():.4f}")

In [ ]:
# Train one frozen XGBoost model on base window
X_train = X_splits[BASE_WINDOW]
y_train = y_splits[BASE_WINDOW]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)

n_neg = max((y_tr == 0).sum(), 1)
n_pos = max((y_tr == 1).sum(), 1)
scale_pos_weight = n_neg / n_pos

model = xgb.XGBClassifier(
    max_depth=6,
    n_estimators=250,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

train_auc = roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1])
val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
print(f"Train AUC: {train_auc:.4f}")
print(f"Val AUC:   {val_auc:.4f}")

In [ ]:
# SHAP attributions for each window
N_EXPLAIN = 300
explainer = shap.TreeExplainer(model)
shap_attributions = {}

for window_name in WINDOW_ORDER:
    X_window = X_splits[window_name]
    n = min(N_EXPLAIN, len(X_window))
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(X_window), size=n, replace=False)
    X_sample = X_window[idx]

    values = explainer.shap_values(X_sample)
    if isinstance(values, list):
        values = values[1]
    shap_attributions[window_name] = values

    top_idx = np.argmax(np.abs(values).mean(axis=0))
    print(f"{window_name}: explained {n} rows, top feature={FEATURE_NAMES[top_idx]}")

In [ ]:
# TESI helpers
def compute_tesi(base_attr: np.ndarray, comp_attr: np.ndarray) -> float:
    base_mean = np.abs(base_attr).mean(axis=0)
    comp_mean = np.abs(comp_attr).mean(axis=0)

    denom = (np.linalg.norm(base_mean) * np.linalg.norm(comp_mean)) + 1e-12
    cosine_sim = float(np.dot(base_mean, comp_mean) / denom)

    rho, _ = spearmanr(base_mean, comp_mean)
    rho = 0.0 if np.isnan(rho) else float(rho)
    spearman_norm = (rho + 1.0) / 2.0

    return 0.5 * cosine_sim + 0.5 * spearman_norm


def bootstrap_tesi_ci(
    base_attr: np.ndarray,
    comp_attr: np.ndarray,
    n_boot: int = 200,
    ci: float = 0.95,
    seed: int = 42,
):
    rng = np.random.RandomState(seed)
    n = len(comp_attr)
    boot_vals = []

    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        boot_vals.append(compute_tesi(base_attr, comp_attr[idx]))

    boot_vals = np.array(boot_vals)
    alpha = 1.0 - ci
    low = float(np.percentile(boot_vals, 100 * alpha / 2))
    high = float(np.percentile(boot_vals, 100 * (1 - alpha / 2)))
    return low, high

print("TESI helper functions ready")

In [ ]:
# Compute AUC and TESI per window
base_attr = shap_attributions[BASE_WINDOW]

rows = []
for window_name in WINDOW_ORDER:
    X_window = X_splits[window_name]
    y_window = y_splits[window_name]

    auc = roc_auc_score(y_window, model.predict_proba(X_window)[:, 1])
    tesi = compute_tesi(base_attr, shap_attributions[window_name])
    ci_low, ci_high = bootstrap_tesi_ci(base_attr, shap_attributions[window_name], n_boot=200, ci=0.95, seed=SEED)

    rows.append({
        "Window": window_name,
        "AUC": auc,
        "TESI": tesi,
        "TESI_CI_Low": ci_low,
        "TESI_CI_High": ci_high,
    })

results = pd.DataFrame(rows)
display(results.round(4))

In [ ]:
# Simple final figure: TESI and AUC over windows
x = np.arange(len(results))

fig, ax1 = plt.subplots(figsize=(10, 5), dpi=140)

tesi = results["TESI"].to_numpy()
ci_low = results["TESI_CI_Low"].to_numpy()
ci_high = results["TESI_CI_High"].to_numpy()

ax1.plot(x, tesi, marker="o", linewidth=2, color="#1f77b4", label="TESI")
ax1.fill_between(x, ci_low, ci_high, color="#1f77b4", alpha=0.2)
ax1.set_ylabel("TESI", color="#1f77b4")
ax1.set_ylim(0.5, 1.02)
ax1.tick_params(axis="y", labelcolor="#1f77b4")

ax2 = ax1.twinx()
auc = results["AUC"].to_numpy()
ax2.plot(x, auc, marker="s", linewidth=2, color="#d62728", label="AUC")
ax2.set_ylabel("AUC", color="#d62728")
ax2.set_ylim(max(0.5, auc.min() - 0.05), min(1.0, auc.max() + 0.05))
ax2.tick_params(axis="y", labelcolor="#d62728")

ax1.set_xticks(x)
ax1.set_xticklabels(results["Window"].tolist())
ax1.set_xlabel("Time Window")
ax1.set_title("Temporal Explanation Stability (TESI) vs Predictive Performance (AUC)")
ax1.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Next Step (Optional)
If you want an even faster run, reduce `N_EXPLAIN` from `300` to `150` and bootstrap iterations in `bootstrap_tesi_ci` from `200` to `100`.